# 10 — Bronze: land raw NHS PROMs into OneLake

**Layer:** Bronze · **Engine:** PySpark · **Write format:** Delta (append-only)

Bronze is the *immutable landing zone*. Nothing is fixed here — no renames, no type
coercion, no filtering. We keep the supplier's column names and the supplier's junk
values (`*`, `9`, `999`, `99999`) exactly as delivered, and we bolt on ingestion
metadata so that any row in gold can be traced back to the file and batch it came from.

That traceability is the whole point under the EU AI Act: *"show me the raw record
behind this clinical decision"* has to be answerable, and Delta time travel is what
makes it answerable at a point in time.

**Prerequisite:** attach this notebook to the demo Lakehouse and upload the raw
NHS PROMs data packs to `Files/landing/proms/`.

## 1. Configuration

In [ ]:
# ---------------------------------------------------------------------------
# Shared configuration. Keep these values identical across notebooks 10-50.
# ---------------------------------------------------------------------------
LAKEHOUSE          = "NHS_PROMs_LH"      # attached (default) lakehouse
LANDING_ROOT       = "Files/data/external"      # raw supplier data packs live here
BRONZE_TABLE       = "bronze.knee_provider"     # append-only raw
PROCEDURE          = "Knee Replacement"

# V-Order + optimized write: smaller files, faster Direct Lake reads downstream.
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize", "1073741824")

# Lakehouse schemas keep the medallion visually separated in the object explorer.
for layer in ("bronze", "silver", "gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {layer}")

print("Configured. Schemas bronze / silver / gold ready.")

## 2. (Optional) unzip the supplier data packs

NHS publishes the PROMs data packs as `.zip`. Spark cannot read a zip directly, so we
unzip once on the driver using the local OneLake mount (`/lakehouse/default/Files/...`),
then let Spark read the resulting CSVs. Idempotent — skips anything already extracted.

Skip this cell if you have already uploaded plain `.csv` files.

In [ ]:
import os, zipfile, glob

local_landing = f"/lakehouse/default/{LANDING_ROOT}"
zips = sorted(glob.glob(f"{local_landing}/**/*Knee*Provider*.zip", recursive=True))
print(f"Found {len(zips)} knee provider zip(s)")

for z in zips:
    target_dir = os.path.join(os.path.dirname(z), "csv")
    os.makedirs(target_dir, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        for member in zf.namelist():
            # NHS zips carry a __MACOSX sidecar; ignore it.
            if member.startswith("__MACOSX") or not member.lower().endswith(".csv"):
                continue
            out = os.path.join(target_dir, os.path.basename(member))
            if os.path.exists(out):
                print(f"  skip (exists): {os.path.basename(out)}")
                continue
            with zf.open(member) as src, open(out, "wb") as dst:
                dst.write(src.read())
            print(f"  extracted: {os.path.basename(out)}")

## 3. Read raw CSV — no schema, no cleaning

`inferSchema=False` is deliberate. Every bronze column lands as `string`. Type decisions
belong in silver where they are documented and testable; inferring them here would mean a
new file with a stray value could silently change the schema of the audit record.

In [ ]:
from pyspark.sql import functions as F

raw = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "false")      # everything lands as string, on purpose
         .option("multiLine", "false")
         .option("mode", "PERMISSIVE")
         .option("columnNameOfCorruptRecord", "_corrupt_record")
         .csv(f"{LANDING_ROOT}/**/*Knee*Provider*.csv")
)

print(f"Raw rows: {raw.count():,}   columns: {len(raw.columns)}")
raw.limit(5).display()

## 4. Add ingestion metadata

Five columns turn an anonymous dataframe into an auditable one:

| column | why it exists |
|---|---|
| `_source_file` | which supplier file this row came from |
| `_ingested_at_utc` | when it entered the estate |
| `_batch_id` | groups one notebook run — the unit you would roll back |
| `_ingest_notebook` | which code produced it |
| `_row_hash` | content fingerprint, used for dedupe on re-runs |

PROMs provider extracts have no patient identifier, so `_row_hash` is also our
de-duplication key — re-running this notebook on the same files does not double-count.

In [ ]:
import uuid

BATCH_ID = str(uuid.uuid4())
print(f"batch_id = {BATCH_ID}")

business_cols = [c for c in raw.columns if not c.startswith("_")]

bronze_df = (
    raw
    .withColumn("_source_file",     F.input_file_name())
    .withColumn("_ingested_at_utc", F.current_timestamp())
    .withColumn("_batch_id",        F.lit(BATCH_ID))
    .withColumn("_ingest_notebook", F.lit("10_bronze_ingest"))
    .withColumn("_row_hash",        F.sha2(F.concat_ws("||", *[F.coalesce(F.col(f"`{c}`"), F.lit("")) for c in business_cols]), 256))
)

bronze_df.select("_source_file", "_batch_id", "_row_hash").limit(3).display()

## 5. Write to Bronze as Delta — append only

**This is the answer to "does my parquet file work?"** — plain parquet works for reading,
but it cannot do what the next three lines do:

* `MERGE` on `_row_hash` gives us **idempotent re-runs** (parquet has no upsert),
* the write is **ACID** — a failed run leaves no half-written table,
* `DESCRIBE HISTORY` / `VERSION AS OF` give **time travel**, i.e. the audit trail,
* the Lakehouse **SQL analytics endpoint** and **Direct Lake** only see Delta tables.

Partitioned by `Year` because that is how the supplier drops data and how we will
reprocess it.

In [ ]:
from delta.tables import DeltaTable

if not spark.catalog.tableExists(BRONZE_TABLE):
    (bronze_df.write
        .format("delta")
        .partitionBy("Year")
        .option("delta.autoOptimize.optimizeWrite", "true")
        .saveAsTable(BRONZE_TABLE))
    print(f"Created {BRONZE_TABLE} with {bronze_df.count():,} rows")
else:
    tgt = DeltaTable.forName(spark, BRONZE_TABLE)
    (tgt.alias("t")
        .merge(bronze_df.alias("s"), "t._row_hash = s._row_hash")
        .whenNotMatchedInsertAll()
        .execute())
    print(f"Merged batch {BATCH_ID} into {BRONZE_TABLE}")

spark.sql(f"SELECT Year, COUNT(*) AS rows FROM {BRONZE_TABLE} GROUP BY Year ORDER BY Year").display()

## 6. The audit trail, on stage

`DESCRIBE HISTORY` is the single most useful thing to put on screen in the governance
segment: every write, who ran it, which operation, how many rows. This is what a plain
parquet folder cannot give you.

In [ ]:
spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}").select(
    "version", "timestamp", "userName", "operation", "operationMetrics"
).display()

In [ ]:
# Time travel: what did bronze look like at version 0?
# (This is the "show me the record as it stood when the model was trained" answer.)
try:
    v0 = spark.read.format("delta").option("versionAsOf", 0).table(BRONZE_TABLE)
    print(f"version 0 row count: {v0.count():,}")
except Exception as e:
    print("Only one version so far:", e)

## 7. Data-quality gate

Bronze does not reject rows — but it *does* report. Anything alarming here is a signal to
stop before silver, not something to quietly fix downstream.

In [ ]:
checks = spark.sql(f"""
    SELECT
        COUNT(*)                                                        AS total_rows,
        COUNT(DISTINCT _row_hash)                                       AS distinct_rows,
        SUM(CASE WHEN `Knee Replacement Pre-Op Q Score` IS NULL
                 OR TRIM(`Knee Replacement Pre-Op Q Score`) = '' THEN 1 ELSE 0 END) AS missing_oks_t0,
        SUM(CASE WHEN `Revision Flag` = '1' THEN 1 ELSE 0 END)           AS revisions,
        COUNT(DISTINCT `Provider Code`)                                 AS providers,
        COUNT(DISTINCT Year)                                            AS years
    FROM {BRONZE_TABLE}
""")
checks.display()

row = checks.first()
assert row["total_rows"] == row["distinct_rows"], "Duplicate _row_hash values in bronze"
print("Bronze OK — no duplicates.")

---

**Next:** `20_silver_clean` — rename to the analytical vocabulary, cast types, turn the
supplier's sentinel codes into real `NULL`s, and drop the records we cannot model.